# 머신러닝이란 — 첫 모델 만들기

> ⏱ 40분 · CPU로 충분 · 선수 지식: 파이썬 기초

**목표:** "데이터로 모델을 학습시킨다"는 것이 무엇인지, scikit-learn으로 10줄짜리 모델을 만들며 전체 흐름을 익힙니다. 이 흐름은 LLM 학습까지 똑같이 이어집니다.

## 프로그래밍과 머신러닝의 차이

- **프로그래밍:** 사람이 규칙을 짠다. `if 꽃잎 길이 > 2.5: ...`
- **머신러닝:** 사람은 **예시(데이터)**를 주고, 규칙은 컴퓨터가 찾는다.

손글씨 숫자를 구분하는 규칙을 `if`문으로 짤 수 있을까요? 거의 불가능합니다. 하지만 "이 그림은 3, 저 그림은 7"이라는 예시는 얼마든지 줄 수 있습니다. 이것이 머신러닝이 필요한 이유입니다.

머신러닝의 모든 작업은 다음 흐름을 따릅니다.

```
데이터 준비 → 학습용/평가용으로 나누기 → 모델 학습(fit) → 처음 보는 데이터로 평가 → 개선
```

## 데이터 살펴보기

8×8 픽셀의 손글씨 숫자 1797장입니다. 이미지 한 장은 숫자 64개짜리 벡터, 정답(라벨)은 0~9입니다.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
X, y = digits.data, digits.target
print(X.shape, y.shape)      # (1797, 64) 입력 , (1797,) 정답
print(X[0].reshape(8, 8))    # 이미지 = 숫자들의 배열
print("정답:", y[0])

fig, axes = plt.subplots(1, 8, figsize=(10, 2))
for ax, img, label in zip(axes, digits.images, y):
    ax.imshow(img, cmap="gray_r"); ax.set_title(label); ax.axis("off")
plt.show()

- **X (입력, feature):** 모델이 보는 것. 샘플 수 × 특징 수의 표.
- **y (정답, label):** 모델이 맞혀야 하는 것.

**코드 읽기**

- `from sklearn.datasets import load_digits` — 왜 scikit-learn인가: 고전 머신러닝 알고리즘 수십 개를 **같은 사용법**(`fit` / `predict` / `score`)으로 제공하는 표준 라이브러리입니다. 딥러닝 프레임워크(PyTorch)는 2부에서 씁니다. 여기서는 "학습의 흐름"만 익히면 되므로 가장 간단한 도구를 고릅니다.
- `load_digits()` — 인터넷 연결 없이 라이브러리 안에 들어 있는 작은 데이터셋. 1797장이라 어떤 컴퓨터에서도 몇 초면 끝납니다. MNIST(2부)의 축소판입니다.
- `digits.data` vs `digits.images` — 같은 데이터를 두 가지 모양으로 담고 있습니다. `data`는 `(1797, 64)`로 펼친 것(모델 입력용), `images`는 `(1797, 8, 8)`(그림 그리기용). 고전 머신러닝 모델은 입력이 **1차원 벡터**여야 하므로 이미지를 한 줄로 펼쳐서 줍니다.
- `X.shape` — `shape`을 찍는 습관은 여기서부터입니다. "샘플 수 × 특징 수"를 확인해야 뒤에서 오류가 났을 때 원인을 찾을 수 있습니다.
- `plt.subplots(1, 8)` — 그림 8개를 가로로 나란히 그릴 칸(`axes`)을 만듭니다. `imshow`가 2차원 배열을 이미지로 그리고, `cmap="gray_r"`는 0을 흰색, 큰 값을 검은색으로 표시합니다(종이에 쓴 글씨처럼 보이게).

## 가장 중요한 습관: 데이터를 나눈다

모델을 학습에 쓴 데이터로 평가하면, 시험 문제를 미리 보여주고 시험을 치르는 것과 같습니다. 반드시 **일부를 떼어 놓고** 학습이 끝난 뒤 그것으로 평가합니다.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
print(len(X_train), "장으로 학습 /", len(X_test), "장으로 평가")

**코드 읽기**

- `train_test_split(X, y, ...)` — X와 y를 **같은 순서로** 섞어서 둘로 나눕니다. 직접 인덱스로 자르면 X와 y의 짝이 어긋나는 실수가 잦아서 이 함수를 씁니다.
- `test_size=0.25` — 25%를 평가용으로 뗍니다. 데이터가 많으면 10%, 적으면 30%까지도 씁니다. 정해진 답은 없고 "평가 결과를 믿을 만큼은 남긴다"가 기준입니다.
- `random_state=0` — 섞는 순서를 고정합니다. 이것이 없으면 실행할 때마다 다른 데이터가 test로 가서, 정확도가 달라진 이유가 **모델 때문인지 분할 때문인지** 알 수 없게 됩니다. 실험을 비교하려면 무작위성은 항상 고정합니다.
- `stratify=y` — 각 숫자(0~9)의 비율이 train과 test에서 같도록 나눕니다. 이것이 없으면 우연히 test에 "8"이 거의 없는 식으로 치우칠 수 있습니다. 분류 문제에서는 습관처럼 넣습니다.
## 첫 모델: k-최근접 이웃 (kNN)

가장 직관적인 모델입니다. 새 그림이 들어오면 **학습 데이터 중 가장 비슷한 k장**을 찾아 다수결로 답합니다.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train, y_train)               # 학습
pred = model.predict(X_test)              # 예측
print("예측:", pred[:10])
print("정답:", y_test[:10])
print("정확도:", model.score(X_test, y_test))

scikit-learn의 모든 모델은 `fit`(학습) → `predict`(예측) → `score`(평가)라는 같은 사용법을 가집니다. 그래서 모델을 바꿔 끼우기가 쉽습니다.

**코드 읽기**

- `KNeighborsClassifier(n_neighbors=3)` — 왜 첫 모델로 kNN인가: 수식이 전혀 없이 "비슷한 것끼리 같은 답"이라는 직관 하나로 동작해서, 학습·예측·평가의 흐름에만 집중할 수 있습니다. `n_neighbors`가 이 모델의 유일한 하이퍼파라미터입니다. 1이면 가장 가까운 한 장만 보므로 노이즈에 민감하고, 너무 크면 멀리 있는 것까지 섞여 둔해집니다.
- `model.fit(X_train, y_train)` — "학습"이지만 kNN은 사실 데이터를 **저장만** 합니다. 진짜 계산은 예측할 때(가까운 이웃 찾기) 일어납니다. 그래서 학습은 빠르고 예측이 느린 모델입니다.
- `model.predict(X_test)` — 테스트 이미지 450장 각각에 대해 답을 돌려줍니다. 반환값은 `y_test`와 같은 모양의 배열이라 바로 비교할 수 있습니다.
- `model.score(X_test, y_test)` — `predict` 결과를 `y_test`와 비교해 맞힌 비율을 냅니다. 분류 모델의 `score`는 정확도, 회귀 모델의 `score`는 R²(다음 레슨)입니다.

## 두 번째 모델: 로지스틱 회귀

kNN은 데이터를 통째로 기억할 뿐 "학습되는 숫자"가 없습니다. 로지스틱 회귀는 다릅니다. 픽셀마다 **가중치**를 두고, `점수 = 픽셀값 × 가중치의 합`으로 각 숫자의 점수를 계산합니다. 학습이란 **정답의 점수가 높아지도록 가중치를 조정하는 과정**입니다. 이것은 층이 하나뿐인 신경망과 같으며, 2부에서 직접 구현합니다.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))   # 입력 크기를 고르게 맞춘 뒤 학습
clf.fit(X_train, y_train)
print("정확도:", clf.score(X_test, y_test))

W = clf[-1].coef_                  # 학습된 가중치: [10개 숫자, 64개 픽셀]
fig, axes = plt.subplots(1, 10, figsize=(12, 1.8))
for d, ax in enumerate(axes):
    ax.imshow(W[d].reshape(8, 8), cmap="bwr"); ax.set_title(d); ax.axis("off")
plt.show()   # 빨강: 이 픽셀이 칠해져 있으면 그 숫자일 가능성 ↑, 파랑: ↓

가중치를 그려 보면 모델이 "0은 가운데가 비어 있다"를 스스로 찾아낸 것이 보입니다.

**코드 읽기**

- `LogisticRegression` — 이름에 "회귀"가 있지만 **분류** 모델입니다. 픽셀 64개 각각에 가중치를 곱해 더한 점수를 클래스마다 계산하고, 점수가 가장 높은 클래스를 답으로 냅니다. 이 "가중치 × 입력의 합"이 2부 신경망의 `nn.Linear`와 정확히 같은 계산이라서, 딥러닝으로 넘어가는 다리로 골랐습니다.
- `max_iter=1000` — 로지스틱 회귀는 kNN과 달리 가중치를 **반복 계산으로 조금씩 고쳐 가며** 찾습니다(2부에서 배울 경사하강법의 사촌). 기본값 100번으로는 다 수렴하지 못했다는 경고가 나오므로 넉넉히 줍니다.
- `StandardScaler()` — 각 특징(픽셀)의 값을 평균 0, 표준편차 1로 맞춥니다. 왜 필요한가: 가중치를 반복으로 찾는 모델은 특징들의 크기가 제각각이면 수렴이 느리거나 불안정합니다. 트리 모델(다음 레슨)은 값의 크기에 영향을 받지 않아 필요 없지만, 선형 모델과 신경망에는 거의 항상 씁니다.
- `make_pipeline(StandardScaler(), LogisticRegression(...))` — 전처리와 모델을 하나로 묶습니다. 왜 묶는가: `fit`할 때 스케일러가 **train 데이터의** 평균·표준편차를 기억하고, `predict`할 때 test 데이터에 **같은 값**을 적용합니다. 따로 하면 test 데이터로 스케일러를 다시 맞추는 실수(정보 누출)를 하기 쉽습니다. 파이프라인은 그 실수를 구조적으로 막아 줍니다.
- `clf[-1].coef_` — 파이프라인의 마지막 단계(모델)에서 학습된 가중치를 꺼냅니다. `coef_`처럼 뒤에 밑줄이 붙은 속성은 "학습으로 정해진 값"이라는 scikit-learn의 약속입니다.
- `cmap="bwr"` — 파랑(음수)–흰색(0)–빨강(양수)으로 그리는 색상표. 가중치의 부호를 보려는 것이므로 회색조 대신 씁니다.

## 정확도만 보면 안 되는 이유

정확도는 "맞힌 비율" 하나로 뭉뚱그린 숫자입니다. **무엇을 무엇으로 틀리는지**는 혼동 행렬이 알려줍니다.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

ConfusionMatrixDisplay.from_predictions(y_test, clf.predict(X_test))
plt.show()
print(classification_report(y_test, clf.predict(X_test)))

- **정밀도(precision):** 모델이 "8"이라고 한 것 중 진짜 8의 비율
- **재현율(recall):** 진짜 8 중에서 모델이 찾아낸 비율

**코드 읽기**

- `ConfusionMatrixDisplay.from_predictions(y_test, pred)` — 정답과 예측을 주면 10×10 표를 그립니다. 행이 실제 숫자, 열이 예측한 숫자입니다. 대각선이 맞힌 개수이고, 대각선 밖의 숫자가 "무엇을 무엇으로 헷갈렸나"입니다. 표를 직접 세어서 그릴 수도 있지만, 축 이름과 색까지 알아서 붙여 주므로 이 함수를 씁니다.
- `classification_report(y_test, pred)` — 클래스별 정밀도·재현율·F1(둘의 조화평균)과 샘플 수(`support`)를 표로 출력합니다. 정확도 하나가 아니라 **클래스별로** 성능을 보고 싶을 때 가장 빠른 방법입니다. 마지막 줄의 `macro avg`는 클래스별 평균, `weighted avg`는 샘플 수로 가중한 평균입니다.

암 진단 모델이라면? 환자 100명 중 1명만 암일 때 "전부 정상"이라고 답해도 정확도는 99%입니다. 그러나 재현율은 0%입니다. **문제에 맞는 지표를 고르는 것**은 모델을 고르는 것만큼 중요합니다.

## 머신러닝 문제의 종류

| 종류 | 정답의 형태 | 예 |
|---|---|---|
| 분류 (classification) | 정해진 범주 중 하나 | 스팸 여부, 숫자 인식, **LLM의 다음 토큰 예측** |
| 회귀 (regression) | 연속된 숫자 | 집값, 기온 예측 |
| 비지도 학습 (unsupervised) | 정답 없음 | 고객 군집화, 차원 축소 |
| 자기지도 학습 (self-supervised) | 데이터 자체에서 정답을 만듦 | LLM 사전학습 (다음 단어가 곧 정답) |

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# 비지도 학습 맛보기: 정답 없이 64차원을 2차원으로 줄여 그려 보기
X2 = PCA(n_components=2).fit_transform(X)
groups = KMeans(n_clusters=10, n_init=10, random_state=0).fit_predict(X)   # 정답을 보지 않고 10개 무리로 나눔
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(X2[:, 0], X2[:, 1], c=y, cmap="tab10", s=8); axes[0].set_title("colored by true digit")
axes[1].scatter(X2[:, 0], X2[:, 1], c=groups, cmap="tab10", s=8); axes[1].set_title("clusters found by KMeans (no labels)")
plt.show()

**코드 읽기**

- `PCA(n_components=2)` — 주성분 분석. 64차원 데이터를 **정보 손실이 가장 적은 2개의 축**으로 눌러 줍니다. 왜 쓰나: 64차원은 그릴 수 없으니, 사람이 볼 수 있는 2차원으로 줄여 데이터가 어떻게 뭉쳐 있는지 눈으로 확인하려는 것입니다. `fit_transform`은 `fit`(축 찾기)과 `transform`(투영)을 한 번에 합니다.
- `KMeans(n_clusters=10)` — 정답을 보지 않고 데이터를 10개 무리로 나눕니다. "가까운 점끼리 같은 무리"라는 규칙으로 중심점 10개를 반복해서 옮깁니다. `n_init=10`은 시작점을 10번 바꿔 가장 좋은 결과를 고르라는 뜻(시작점에 따라 결과가 달라지는 알고리즘이라서), `random_state=0`은 재현을 위한 고정입니다.
- 왜 `fit_predict(X)`를 원본 64차원에 하고 2차원 그림에만 PCA를 쓰나: 군집은 정보가 온전한 원본에서 찾는 것이 정확하고, 그림은 보기 위한 것이기 때문입니다. 오른쪽 그림의 색이 왼쪽과 비슷하게 갈라진다면, **정답 없이도** 데이터 자체에 구조가 있다는 뜻입니다.

## 핵심 정리

- 머신러닝은 **규칙 대신 예시**를 주고 컴퓨터가 규칙(파라미터)을 찾게 하는 것입니다.
- 흐름은 언제나 `데이터 → 분할 → 학습(fit) → 처음 보는 데이터로 평가`입니다.
- 학습에 쓴 데이터로 평가하면 안 됩니다. 테스트 데이터는 마지막까지 떼어 둡니다.
- 정확도 하나만 믿지 말고 혼동 행렬, 정밀도, 재현율로 **어떻게 틀리는지** 보세요.
- 로지스틱 회귀의 "가중치 × 입력의 합"은 신경망의 가장 작은 단위입니다.

## 스스로 점검

답을 머릿속으로 먼저 말해 본 뒤 펼쳐 보세요.

<details><summary>Q1. kNN에서 k=1일 때 <b>학습 데이터</b>에 대한 정확도는 얼마일까요? 그것이 좋은 모델이라는 뜻일까요?</summary>

100%입니다. 자기 자신이 가장 가까운 이웃이기 때문입니다. 하지만 이것은 외운 것일 뿐이며, 모델의 실력은 테스트 데이터로만 알 수 있습니다.

</details>

<details><summary>Q2. 사기 거래가 0.1%인 데이터에서 정확도 99.9%인 모델은 좋은 모델일까요?</summary>

알 수 없습니다. "전부 정상"이라고만 답해도 99.9%가 나옵니다. 이런 불균형 데이터에서는 사기 거래에 대한 재현율과 정밀도를 봐야 합니다.

</details>

<details><summary>Q3. LLM의 다음 토큰 예측은 위 표의 어느 종류에 해당하나요?</summary>

모델이 푸는 문제의 형태는 **분류**(수만 개 토큰 중 하나 고르기)이고, 정답을 사람이 붙이지 않고 텍스트 자체에서 얻는다는 점에서 **자기지도 학습**입니다.

</details>

## 직접 고쳐보기

1. `n_neighbors`를 1, 5, 15, 50으로 바꿔 테스트 정확도를 비교해 보세요.
2. `test_size=0.9`로 바꿔(학습 데이터를 10%만 사용) 두 모델을 다시 학습시켜 보세요. 데이터 양이 성능에 미치는 영향이 보입니다.
3. `random_state`를 바꾸면 정확도가 조금씩 달라집니다. 왜일까요? 그렇다면 "모델 A가 0.5% 더 좋다"는 결론은 언제 믿을 수 있을까요?
4. 혼동 행렬에서 가장 많이 헷갈리는 숫자 쌍을 찾고, 그 틀린 이미지들을 `plt.imshow`로 직접 그려 보세요. 사람이 봐도 헷갈리나요?
5. (도전) `from sklearn.svm import SVC`나 `from sklearn.ensemble import RandomForestClassifier`로 모델만 바꿔 보세요. 나머지 코드는 그대로입니다.

<details><summary>힌트와 예상 결과 — 먼저 스스로 해 본 뒤 펼치세요</summary>

1. k=1 약 0.98, k=5 약 0.98, k=15 약 0.97, k=50 약 0.95. k가 커지면 멀리 있는 다른 숫자까지 다수결에 끼어 조금씩 떨어집니다. 이 데이터에서는 작은 k가 유리합니다(클래스가 잘 뭉쳐 있어서).
2. 학습 180장으로도 kNN 약 0.9, 로지스틱 회귀 약 0.85 근처가 나옵니다. 데이터가 1/7로 줄면 정확도가 몇 %p 떨어지지만 생각보다 버팁니다. 이 데이터가 쉽기 때문이고, 어려운 문제일수록 데이터 양에 민감합니다.
3. 어떤 샘플이 test로 가느냐에 따라 ±1%p쯤 흔들립니다. 그러므로 0.5%p 차이는 우연일 수 있습니다. 여러 `random_state`로 반복해 평균과 편차를 보거나, 교차검증(다음 레슨)으로 판단합니다.
4. 보통 8↔1, 8↔9, 3↔5, 4↔9가 헷갈립니다. `wrong = np.where((y_test == 8) & (pred == 1))[0]`로 골라 그리면 사람이 봐도 애매한 글씨가 많습니다. 모델의 오류가 데이터의 한계와 겹치는지 확인하는 습관입니다.
5. `SVC()` 약 0.98~0.99, `RandomForestClassifier()` 약 0.97. `fit/predict/score` 세 줄만 같으면 모델을 갈아 끼울 수 있다는 것이 요점입니다.

</details>